# Wasserstein Geodesics Between Gaussian Measures

For Gaussian laws and the quadratic cost, the optimal map is affine.  This makes the Wasserstein geodesic explicit: means move linearly and covariances follow
$$
    \Sigma_t=((1-t)I+tA)\Sigma_0((1-t)I+tA),
    \qquad
    A\Sigma_0A=\Sigma_1,
$$
with $A$ symmetric positive.  The figure shows the one-dimensional formula and the two-dimensional covariance ellipse evolution.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY,
    DIRAC_MARKER_SIZE, setup_matplotlib, figure_dir, save_pdf,
    remove_axes, box_axes, padded_limits, interp_color, draw_point_clouds,
)

setup_matplotlib()

from matplotlib.patches import Ellipse

NAME = "monge-gaussian-w2-geodesic"
OUT = figure_dir(NAME)

def sqrtm_spd(S):
    w, V = np.linalg.eigh(S)
    return (V * np.sqrt(np.maximum(w, 0))) @ V.T

def invsqrtm_spd(S):
    w, V = np.linalg.eigh(S)
    return (V * (1 / np.sqrt(np.maximum(w, 1e-15)))) @ V.T

def ellipse_patch(mean, cov, color, alpha=0.85, lw=1.25):
    w, V = np.linalg.eigh(cov)
    order = np.argsort(w)[::-1]
    w = w[order]
    V = V[:, order]
    theta = np.degrees(np.arctan2(V[1, 0], V[0, 0]))
    return Ellipse(mean, width=2*np.sqrt(w[0]), height=2*np.sqrt(w[1]), angle=theta, fill=False, lw=lw, color=color, alpha=alpha)

In dimension one, the standard deviation is the linear coordinate for $\Wass_2$ on Gaussians.

In [ ]:
x = np.linspace(-3.2, 3.6, 500)
m0, s0 = -1.10, 0.42
m1, s1 = 1.25, 0.92
ts = [0, 0.25, 0.5, 0.75, 1]
fig, ax = plt.subplots(figsize=(3.1, 1.85))
for t in ts:
    m = (1-t) * m0 + t * m1
    s = (1-t) * s0 + t * s1
    rho = np.exp(-0.5*((x-m)/s)**2) / (np.sqrt(2*np.pi)*s)
    ax.plot(x, rho, color=interp_color(t), lw=1.55 if t in [0,1] else 1.2, alpha=0.95)
box_axes(ax)
ax.set_xlim(x.min(), x.max())
ax.set_ylim(0, 1.02)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$\rho_t(x)$")
save_pdf(fig, OUT / "density-1d.pdf", pad_inches=0.055)
plt.close(fig)

In two dimensions, the same formula is visible through covariance ellipses.  The line of centers is Euclidean; the ellipses rotate and change anisotropy through the Bures--Wasserstein interpolation.

In [ ]:
S0 = np.array([[0.55, 0.22], [0.22, 0.24]])
S1 = np.array([[0.30, -0.18], [-0.18, 0.82]])
mu0 = np.array([-1.15, -0.25])
mu1 = np.array([1.25, 0.32])
S0h = sqrtm_spd(S0)
A = invsqrtm_spd(S0) @ sqrtm_spd(S0h @ S1 @ S0h) @ invsqrtm_spd(S0)
fig, ax = plt.subplots(figsize=(3.0, 2.25))
for t in ts:
    B = (1-t) * np.eye(2) + t * A
    S = B @ S0 @ B.T
    mu = (1-t) * mu0 + t * mu1
    ax.add_patch(ellipse_patch(mu, S, interp_color(t), lw=1.35 if t in [0,1] else 1.05))
    ax.scatter([mu[0]], [mu[1]], s=DIRAC_MARKER_SIZE, color=interp_color(t), marker="o", edgecolor="none", linewidth=0, zorder=3)
ax.plot([mu0[0], mu1[0]], [mu0[1], mu1[1]], color=LIGHT_GRAY, lw=0.8, zorder=0)
ax.set_aspect("equal")
ax.set_xlim(-2.1, 2.1)
ax.set_ylim(-1.25, 1.35)
remove_axes(ax)
save_pdf(fig, OUT / "ellipses-2d.pdf", pad_inches=0.06)
plt.close(fig)